# LTV PostgreSQL Integration

This notebook loads the final LTV-based customer datasets into PostgreSQL so they can be used later for dashboarding and business reporting.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import os

In [2]:
load_dotenv()

True

In [3]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

In [4]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database connection created successfully.")

Database connection created successfully.


In [5]:
pd.read_sql("SELECT version();", engine)

,version
0,"PostgreSQL 18.3 on x86_64-windows, compiled by..."


*A PostgreSQL connection was successfully established using SQLAlchemy and environment variables. The connection was verified by executing a test SQL query.*

In [7]:
df = pd.read_csv(
    "../data/processed/customer_churn_ltv_final.csv"
)

df.shape

(7032, 34)

In [9]:
df.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'PaperlessBilling',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_One year',
 'Contract_Two year',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check',
 'Estimated_LTV',
 'LTV_Segment',
 'Customer_Priority']

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 34 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7032 non-null   int64  
 1   SeniorCitizen                          7032 non-null   int64  
 2   Partner                                7032 non-null   int64  
 3   Dependents                             7032 non-null   int64  
 4   tenure                                 7032 non-null   int64  
 5   PhoneService                           7032 non-null   int64  
 6   PaperlessBilling                       7032 non-null   int64  
 7   MonthlyCharges                         7032 non-null   float64
 8   TotalCharges                           7032 non-null   float64
 9   Churn                                  7032 non-null   int64  
 10  MultipleLines_No phone service         7032 non-null   int64  
 11  MultipleLines_Y

In [11]:
df.to_sql(
    "customer_churn_ltv",
    engine,
    if_exists="replace",
    index=False
)

print("Table created successfully.")

Table created successfully.


In [13]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM customer_churn_ltv;",
    engine
)

,total_rows
0,7032


The final customer churn and LTV dataset was successfully loaded into PostgreSQL and will serve as the primary data source for business analytics, dashboarding, and reporting.

In [14]:
pd.read_sql(
    "SELECT * FROM customer_churn_ltv LIMIT 5;",
    engine
)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Estimated_LTV,LTV_Segment,Customer_Priority
0,1,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,1,0,29.85,Low Value,Low Value - Low Risk
1,0,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,1,0,0,0,1,1936.30,Medium Value,Medium Value - Low Risk
2,0,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,0,1,107.70,Low Value,Low Value - High Risk
3,0,0,0,0,45,0,0,42.30,1840.75,0,...,0,0,1,0,0,0,0,1903.50,Medium Value,Medium Value - Low Risk
4,1,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,0,1,0,141.40,Low Value,Low Value - High Risk


In [15]:
# LTV segment counts
pd.read_sql(
    """
    SELECT "LTV_Segment", COUNT(*) AS customer_count
    FROM customer_churn_ltv
    GROUP BY "LTV_Segment"
    ORDER BY customer_count DESC;
    """,
    engine
)

,LTV_Segment,customer_count
0,Low Value,3516
1,High Value,1758
2,Medium Value,1758


In [16]:
# Customer Priority counts
pd.read_sql(
    """
    SELECT "Customer_Priority", COUNT(*) AS customer_count
    FROM customer_churn_ltv
    GROUP BY "Customer_Priority"
    ORDER BY customer_count DESC;
    """,
    engine
)

,Customer_Priority,customer_count
0,Low Value - Low Risk,2306
1,High Value - Low Risk,1504
2,Medium Value - Low Risk,1353
3,Low Value - High Risk,1210
4,Medium Value - High Risk,405
5,High Value - High Risk,254


In [17]:
# high-priority customers CSV
high_priority_df = pd.read_csv(
    "../data/processed/high_priority_customers.csv"
)

high_priority_df.shape

(254, 34)

In [18]:
high_priority_df.head(2)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Estimated_LTV,LTV_Segment,Customer_Priority
0,0,0,0,0,49,1,1,103.70,5036.30,1,...,0,1,0,0,0,0,0,5081.30,High Value,High Value - High Risk
1,0,0,1,1,47,1,1,99.35,4749.15,1,...,0,1,0,0,0,1,0,4669.45,High Value,High Value - High Risk


In [19]:
# Saving it to PostgreSQL
high_priority_df.to_sql(
    "high_priority_customers",
    engine,
    if_exists="replace",
    index=False
)

print("High priority customers table created successfully.")

High priority customers table created successfully.


In [20]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM high_priority_customers;",
    engine
)

,total_rows
0,254


In [22]:
# Preview high-priority records
pd.read_sql(
    """
    SELECT "Estimated_LTV", "LTV_Segment", "Customer_Priority", "Churn"
    FROM high_priority_customers
    LIMIT 10;
    """,
    engine
)

,Estimated_LTV,LTV_Segment,Customer_Priority,Churn
0,5081.30,High Value,High Value - High Risk,1
1,4669.45,High Value,High Value - High Risk,1
2,7480.00,High Value,High Value - High Risk,1
3,5321.25,High Value,High Value - High Risk,1
4,5027.05,High Value,High Value - High Risk,1
5,5462.60,High Value,High Value - High Risk,1
6,4452.30,High Value,High Value - High Risk,1
7,5154.40,High Value,High Value - High Risk,1
8,4497.80,High Value,High Value - High Risk,1
9,6514.20,High Value,High Value - High Risk,1


*The high-priority customer dataset was successfully loaded into PostgreSQL as a separate table. This table contains the most important retention customers and can be directly used for campaign planning, dashboarding, and business decision-making.*

The high-priority customer table was successfully created and verified in PostgreSQL. All records belong to the High Value - High Risk segment, representing customers who generate significant revenue while also showing a high likelihood of churn. This table can be used directly for retention campaigns, customer outreach, and business decision-making.